In [ ]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(df.drop('target', axis=1),
                                                    df['target'], test_size=0.2, random_state=101, stratify=df['target'])

#initial dimensions and distribution of classes of train and test dataset
print(X_train.shape)
print(X_test.shape)
print(y_train.value_counts())
print(y_test.value_counts())

In [ ]:
#create a Random object so that it only affects whichever other objects uses it
import random
myrandom = random.Random(101)

#Performing random undersampling

removed_index = myrandom.sample(sorted(y_train[y_train==0].index), 22000) #extracting the index of non-fraud rows that we want to remove from the training dataset
removed_X = X_train.loc[removed_index]
reduced_X = X_train.drop(removed_index)
removed_y = y_train[removed_index]
reduced_y = y_train.drop(removed_index)
new_X_test = pd.concat([X_test,removed_X]) #adding the index of rows that were removed into the test dataset
new_y_test = pd.concat([y_test,removed_y])

#dimensions and distributions of classes after undersampling
print(reduced_X.shape)
print(new_X_test.shape)
print(reduced_y.value_counts())

#distribution of classes in training dataset
plt.bar([0,1], reduced_y.value_counts(), tick_label=[0,1])

#distribution of classes in test dataset
plt.bar([0,1], new_y_test.value_counts(), tick_label=[0,1])

#checking the data distribution of each column to check for outliers as well as the magnitude in each column
reduced_X.describe()

#Use standardization/MinMaxScaler as some cols have outliers such as the consumption_levels
from sklearn.preprocessing import MinMaxScaler
minmax_scaler = MinMaxScaler()
X_train_scaled = minmax_scaler.fit_transform(reduced_X)
X_test_scaled = minmax_scaler.transform(new_X_test)

In [2]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

In [ ]:
# Model Architecture Building
model = Sequential([
    Input(shape=(16,)),                       # 16 input features
    Dense(40, activation='relu'),
    Dropout(0.1),
    
    Dense(40, activation='relu'),
    Dropout(0.1),
    
    Dense(20, activation='relu'),
    Dropout(0.1),
    
    Dense(1, activation='sigmoid')            # Binary classification output
])

In [ ]:
model.summary()

In [ ]:
# Model Compilation
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',               # Appropriate for 0/1 fraud label
    metrics=[
        'accuracy',
        tf.keras.metrics.Precision(name='precision'),
        tf.keras.metrics.Recall(name='recall'),
        tf.keras.metrics.AUC(name='auc')
    ]
)

# Define Early Stopping
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True
)

In [ ]:
# ✅ Optional: Handle Class Imbalance (if fraud cases are rare)
# For example, if fraud:non-fraud = 1:100, you can compute weights like this:
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_train),
    y=y_train
)
class_weights = dict(enumerate(class_weights))
print("Class Weights:", class_weights)

In [ ]:
# Train Model
history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=50,
    batch_size=256,
    class_weight=class_weights,
    callbacks=[early_stop],
    verbose=1
)

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.metrics import recall_score

In [ ]:
NN_pred = model.predict(X_test_scaled)

nn_pred = (NN_pred < 0.5).astype("int").flatten()

print(confusion_matrix(new_y_test, nn_pred))
print(classification_report(new_y_test, nn_pred))

In [ ]:
# Evaluate model performance
test_results = model.evaluate(X_test, y_test, verbose=0)
print(f"Test Accuracy: {test_results[1]:.4f}")
print(f"Precision: {test_results[2]:.4f}")
print(f"Recall: {test_results[3]:.4f}")
print(f"AUC: {test_results[4]:.4f}")

In [ ]:
from sklearn.metrics import roc_curve, precision_recall_curve, auc
import matplotlib.pyplot as plt

# Predict probabilities
y_pred_proba = model.predict(X_test).ravel()

# ROC Curve
fpr, tpr, _ = roc_curve(y_test, y_pred_proba)
roc_auc = auc(fpr, tpr)
plt.figure(figsize=(6,5))
plt.plot(fpr, tpr, label=f'ROC curve (AUC = {roc_auc:.3f})')
plt.plot([0, 1], [0, 1], 'k--')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver Operating Characteristic')
plt.legend()
plt.show()

# Precision–Recall Curve
precision, recall, _ = precision_recall_curve(y_test, y_pred_proba)
plt.figure(figsize=(6,5))
plt.plot(recall, precision)
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('Precision–Recall Curve')
plt.show()